## BERT and the transformers library

* the transformers library allows you to access pretrained BERT vectors
* people don't really train BERT from scratch -- it's a very large model and you'd need a lot of data to get reliable representations
* however, you can fine-tune BERT on your domain (beyond this notebook)
* documentation: https://huggingface.co/docs/transformers/en/index

## basic retrieval of final-layer vectors with BERT

* often people use the final layers of BERT (the most integrative one) as the representation of word tokens, sentences, or documents.

In [ ]:
!pip install transformers
import transformers
import numpy as np

In [ ]:
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
model = BertModel.from_pretrained("bert-base-cased")

### tokenization

* the vocabulary in BERT is a long array
* as such, we have to transform the strings of words into arrays of numbers
* so, some things we consider words will be represented by a single numerical token, others by two or three

In [ ]:
text = "Replace me by any text you'd like."
encoded_input = tokenizer(text, return_tensors='pt')

In [ ]:
encoded_input

In [ ]:
encoded_input['input_ids']

In [ ]:
encoded_input['input_ids'].numpy()

In [ ]:
encoded_input['input_ids'].numpy()[0]

In [ ]:
for i in encoded_input['input_ids'].numpy()[0]:
    print(i, tokenizer.decode(i))

### computing contextual vectors

In [ ]:
output = model(**encoded_input)

In [ ]:
encoded_input['input_ids'].shape, output['last_hidden_state'].shape

In [ ]:
M = output['last_hidden_state'].detach().numpy()

In [ ]:
M[:,11,:][0]

### retrieving a specific token

a pipeline we will see more of:
* retrieve a particular token with a `next()` function.
* note that this is a heuristic approach; you would need a stronger heuristic for longer sentences (that may contain repeated occurrences of the same target word)
* put the resulting contextual vectors for the target words in an np matrix

In [ ]:
# option 1: building a list of vectors, and transforming it to an np.array at the end

sentences = ['The dog can bark loudly.', 
             'He scraped the bark off.',
             'The bark of this tree is not healthy.']
bark_vectors = []
for s in sentences:
    encoded_input = tokenizer(s, return_tensors='pt')
    bark_token = next((i for i,encoding in enumerate(encoded_input['input_ids'][0])
                       if tokenizer.decode([encoding]).startswith('bark')),None)
    print(bark_token, s)
    output = model(**encoded_input)
    vectors = output['last_hidden_state'].detach().numpy()
    bark_vector = vectors[:,bark_token,:][0]
    bark_vectors.append(bark_vector)
bark_vectors = np.array(bark_vectors)
print(bark_vectors.shape)

In [ ]:
# option 2: with a pre-declared bark_vectors (if you know the number of sentences in advance)

sentences = ['The dog can bark loudly.', 
             'He scraped the bark off.',
             'The bark of this tree is not healthy.']
bark_vectors = np.zeros((len(sentences), 768)) 
# creates an N-by-768 dimensional matrix filled with zeros
# 768 is the dimensionality of bert vectors
#
for si,s in enumerate(sentences):
    encoded_input = tokenizer(s, return_tensors='pt')
    bark_token = next((i for i,encoding in enumerate(encoded_input['input_ids'][0])
                       if tokenizer.decode([encoding]).startswith('bark')),None)
    print(bark_token, s)
    output = model(**encoded_input)
    vectors = output['last_hidden_state'].detach().numpy()
    bark_vector = vectors[:,bark_token,:][0]
    bark_vectors[si] += bark_vector # <== here we add the token vector to the matrix, for the right index 'si'
print(bark_vectors.shape)

### cosine similarity

this allows you to generate a distance matrix between all word tokens, something that will come in handy!

In [ ]:
!pip install scikit-learn
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(bark_vectors)

## lexical substitution

with the masked token prediction pipeline

In [ ]:
text = "I got [MASK] and I could no longer be friends with him."

In [ ]:
!pip install torch
import torch
from transformers import BertForMaskedLM

In [ ]:
model = BertForMaskedLM.from_pretrained('bert-base-cased')

# step 1: encode the sentence into tokens
input_items = tokenizer(text, return_tensors='pt')
mask_token_ix = next((i for i,e in enumerate(input_items.input_ids[0]) if e == tokenizer.mask_token_id),None)
print('masked token:', mask_token_ix)
print('encoded input items:', input_items)

In [ ]:
# step 2: create BERT encoding and retrieve logits (distributions over the token vocab)
with torch.no_grad():
    outputs = model(**input_items)
    predictions = outputs.logits 
print(predictions.shape)

In [ ]:
# step 3: 
top_k = 15
print(predictions[0,mask_token_ix].shape)
top_k_tokens = torch.topk(predictions[0,mask_token_ix], top_k)
print(top_k_tokens)
predicted_words = {tokenizer.decode([tok_id]): (-val.item()) for val, tok_id in zip(*top_k_tokens)}
predicted_words

In [ ]:
# post-processing: these numbers come in logits, which can be
# transformed into probabilities with the softmax function
from torch.nn import functional as F

# convert logit score to torch array
torch_logits = torch.from_numpy(predictions[0,mask_token_ix].detach().numpy())

# get probabilities using softmax from logit score and convert it to numpy array
probabilities_scores = F.softmax(torch_logits, dim = -1).numpy()
probabilities_scores.shape

In [ ]:
for i in probabilities_scores.argsort()[-15:][::-1]: # the highest 15 probabilities, reversed so high-to-low
    print(i, tokenizer.decode([i]), probabilities_scores[i])

In [ ]:
# a more pre-fabricated pipeline available through transfromers

from transformers import pipeline
text = text = "I got [MASK] and I could no longer be friends with him."
# text = "In the cookie jar there were five [MASK]."
ppl = pipeline('fill-mask', model = 'google-bert/bert-base-cased', device=-1) # no gpu
results = ppl(text, top_k=15)
for result in results:
    print(result['token_str'], result['score']) 

## lexical substitution for non-masked tokens

* however, this nice pre-fab pipeline only works for **masked** and not for **non-masked** token prediction
* to find replacements for tokens that are not masked, we have to use the more finegrained procedure
* except now we retrieve the position of the target word token instead of the masked one

In [ ]:
# step 1: encoding
text = "I got mad and I could no longer be friends with him."
input_items = tokenizer(text, return_tensors='pt')
#
tw = 'mad'
tw_token = tokenizer(tw)
print('tokenizing the target word tw:', tw_token)
tw_token = tokenizer(tw).input_ids[1]
print('isolating the first token of the target word:', tw_token)
sub_token_ix = next((i for i,e in enumerate(input_items.input_ids[0]) if e == tw_token),None)
print('target word token is:', sub_token_ix)

In [ ]:
# step 2: get output
with torch.no_grad():
    outputs = model(**input_items)
    predictions = outputs.logits # <== distributions over the token vocabulary
#
# convert logit score to torch array
torch_logits = torch.from_numpy(predictions[0,sub_token_ix].detach().numpy())
# get probabilities using softmax from logit score and convert it to numpy array
probabilities_scores = F.softmax(torch_logits, dim = -1).numpy()
#
top_k = 15
top_k_tokens = torch.topk(predictions[0,sub_token_ix], top_k)
print([tok_id for tok_id in top_k_tokens[1]])
predicted_words = {tokenizer.decode([tok_id]): (-val.item()) for val, tok_id in zip(*top_k_tokens)}
predicted_words

now we can wrap it in a function

In [ ]:
def retrieve_substitutions(sentence, target_word, topk=10):
    # step 1: encoding
    input_items = tokenizer(sentence, return_tensors='pt')
    tw_token = tokenizer(target_word).input_ids[1]
    sub_token_ix = next((i for i,e in enumerate(input_items.input_ids[0]) if e == tw_token),None)
    # step 2: get output
    with torch.no_grad():
        outputs = model(**input_items)
        predictions = outputs.logits # <== distributions over the token vocabulary
    #
    # convert logit score to torch array
    torch_logits = torch.from_numpy(predictions[0,sub_token_ix].detach().numpy())
    # get probabilities using softmax from logit score and convert it to numpy array
    probabilities_scores = F.softmax(torch_logits, dim = -1).numpy()
    #
    top_k_tokens = torch.topk(predictions[0,sub_token_ix], topk)
    predicted_words = {tokenizer.decode([tok_id]): probabilities_scores[tok_id] for tok_id in top_k_tokens[1]}
    return predicted_words    

In [ ]:
text = "I got mad and I could no longer be friends with him."
retrieve_substitutions(text, 'mad')

In [ ]:
text = "Sometimes it seems like the whole world has gone mad when I look at the news."
retrieve_substitutions(text, 'mad')

In [ ]:
text = "I got angry and I could no longer be friends with him."
retrieve_substitutions(text, 'angry')

In [ ]:
text = "I got worried and I could no longer be friends with him."
retrieve_substitutions(text, 'worried')

In [ ]:
# notice that it generalizes to masked token prediction!
text = "I got [MASK] and I could no longer be friends with him."
retrieve_substitutions(text, '[MASK]')